**Description**

This notebook is dedicated to merge the CSVs from Dune queries into one final dataset for each Balancer version. To do that, EDA process will be aplicated aim remove unuseful and noise data for future AI model training.

*Final table schema:*

| Column Name                  | Source | Calculation / Logic |
|-----------------------------|--------|----------------------|
| pool_id                     | Raw    | Balancer Vault contract address + ID. |
| launch_date                 | Raw    | `evt_block_time` of the first `joinPool` or `swap`. |
| duration_h                  | Calc   | `(endTime - startTime) / 3600`. |
| start_weight_proj           | Raw    | Initial project token weight (e.g., `0.98`). |
| end_weight_proj             | Raw    | Target project token weight (e.g., `0.50`). |
| weight_slope                | Calc   | `(start_weight - end_weight) / duration_h`. Measures the weight decay "speed". |
| initial_fdv_usd             | Calc   | `Starting_Price * Total_Supply`. Starting Fully Diluted Valuation. |
| total_swaps                 | Raw    | `count(*)` from `balancer_v2_ethereum.evt_Swap` for this pool. |
| unique_users                | Raw    | `count(distinct sender)`. |
| activity_entropy            | Calc   | Shannon entropy of trades over time. High = steady activity; low = single burst. |
| swaps_per_hour              | Calc   | `total_swaps / duration_h`. |
| peak_activity_time          | Calc   | Normalized time (`0–1`) when the highest number of trades occurred. |
| bot_volume_pct              | Calc   | Percentage of volume from addresses with `>10` transactions in the same LBP pool. |
| price_efficiency            | Calc   | `1 - (abs(Final_LBP_Price - Market_Price_24h_Post) / Final_LBP_Price)`. |
| max_drawdown                | Calc   | Largest percentage drop from the programmed price curve during the sale. |
| buy_pressure_avg            | Calc   | Average of `(Actual_Price_t / Theoretical_Price_t)`. |
| price_found_success         | Calc   | `abs(LBP_Final_Price - Market_Price_24h) < 15%`. |
| consistent_activity_success | Calc   | `activity_entropy > 0.7`. Indicates steady trading. |
| not_bot_success             | Calc   | `bot_volume_pct < 30%`. Prevents wallet concentration. |
| final_success               | Calc   | Logical AND of all success conditions above (“Goldilocks” launch). |


In [21]:
expected_columns = [
  'pool_id', 
  'launch_date',
  'duration_h',
  'start_weight_proj',
  'end_weight_proj',
  'weight_slope',
  'initial_fdv_usd',
  'total_swaps',
  'unique_users',
  'activity_entropy',
  'swaps_per_hour',
  'peak_activity_time',
  'bot_volume_pct',
  'price_efficiency',
  'max_drawdown',
  'buy_pressure_avg',
  'price_found_success',
  'consistent_activity_success',
  'not_bot_success',
  'final_success'
]
print(len(expected_columns))

20


# Imports

In [6]:
import pandas as pd

# Merge CSVs

In [7]:
# Format: c_example_column indicates a single column (it probably contains a pool id column too)
#         d_example_dataset indicates a dataset with multiple columns to be merged

dfs = {
  'v1' : {
    'c_price_efficiency': pd.read_csv('../media/gbr/v1/LBP_v1_Price_Efficency.csv'),
    'd_general_metrics': pd.read_csv('../media/gbr/v1/V1_LBP_general_metrics.csv'),
    'c_bot_volume_pct': pd.read_csv('../media/kvz/v1/LBPs_BalancerV1_bot_volume_pct.csv'),
    'c_initial_fdv_usd': pd.read_csv('../media/kvz/v1/LBPs_BalancerV1_initial_fdv_usd.csv')
  },
  'v2' : {
    'c_buy_pressure_avg': pd.read_csv('../media/gbr/v2/LBP_V2_Pools_buy_pressure_avg.csv'),
    'c_price_efficiency': pd.read_csv('../media/gbr/v2/LBP_v2_Price_Efficency.csv'),
    'd_general_metrics': pd.read_csv('../media/gbr/v2/V2_LBP_general_metrics.csv'),
    'c_bot_volume_pct': pd.read_csv('../media/kvz/v2/LBPs_BalancerV2_bot_volume_pct.csv'),
    'c_initial_fdv_usd': pd.read_csv('../media/kvz/v2/LBPs_BalancerV2_initial_fdv_usd.csv')
  }
}

In [11]:
# Print all dataframes shapes

def print_dfs_shapes(dfs):
  for version, datasets in dfs.items():
      print(f"Version: {version}")
      for name, df in datasets.items():
          print(f"  Dataset: {name}, Shape: {df.shape}")

print_dfs_shapes(dfs)

Version: v1
  Dataset: c_price_efficiency, Shape: (76, 8)
  Dataset: d_general_metrics, Shape: (96, 34)
  Dataset: c_bot_volume_pct, Shape: (71, 2)
  Dataset: c_initial_fdv_usd, Shape: (46, 2)
Version: v2
  Dataset: c_buy_pressure_avg, Shape: (47, 19)
  Dataset: c_price_efficiency, Shape: (794, 11)
  Dataset: d_general_metrics, Shape: (338, 37)
  Dataset: c_bot_volume_pct, Shape: (49, 2)
  Dataset: c_initial_fdv_usd, Shape: (47, 3)


In [14]:
# Clean debug columns

# to price_efficiency column keep only 'pool_id' and 'price_efficiency'
dfs['v1']['c_price_efficiency'] = dfs['v1']['c_price_efficiency'][['pool_id', 'price_efficiency']]
dfs['v2']['c_price_efficiency'] = dfs['v2']['c_price_efficiency'][['pool_id', 'price_efficiency']]

# to buy_pressure_avg column keep only 'pool_id' and 'buy_pressure_avg'
dfs['v2']['c_buy_pressure_avg'] = dfs['v2']['c_buy_pressure_avg'][['pool_id', 'buy_pressure_avg']]

# to initial_fdv_usd column keep only 'pool_id' and 'initial_fdv_usd'
dfs['v2']['c_initial_fdv_usd'] = dfs['v2']['c_initial_fdv_usd'][['pool_id', 'initial_fdv_usd']]

print_dfs_shapes(dfs)

Version: v1
  Dataset: c_price_efficiency, Shape: (76, 2)
  Dataset: d_general_metrics, Shape: (96, 34)
  Dataset: c_bot_volume_pct, Shape: (71, 2)
  Dataset: c_initial_fdv_usd, Shape: (46, 2)
Version: v2
  Dataset: c_buy_pressure_avg, Shape: (47, 2)
  Dataset: c_price_efficiency, Shape: (794, 2)
  Dataset: d_general_metrics, Shape: (338, 37)
  Dataset: c_bot_volume_pct, Shape: (49, 2)
  Dataset: c_initial_fdv_usd, Shape: (47, 2)


In [15]:
# What are the common columns between all dataframes in each version? (it should be 'pool_id' only)

for version, datasets in dfs.items():
    common_cols = set(datasets[list(datasets.keys())[0]].columns)
    for name, df in datasets.items():
        common_cols = common_cols.intersection(set(df.columns))
    print(f"Version: {version}, Common Columns: {common_cols}")

Version: v1, Common Columns: {'pool_id'}
Version: v2, Common Columns: {'pool_id'}


In [16]:
# Create a final table with the common pools ids between all datasets of each version, merging all datasets on 'pool_id'

final_dfs = {}
for version, datasets in dfs.items():
    merged_df = None
    for name, df in datasets.items():
        if merged_df is None:
            merged_df = df
        else:
            merged_df = pd.merge(merged_df, df, on='pool_id', how='inner')
    final_dfs[version] = merged_df

print("\nFinal Merged DataFrames Shapes:")
for version, df in final_dfs.items():
    print(f"Version: {version}, Merged Shape: {df.shape}")


Final Merged DataFrames Shapes:
Version: v1, Merged Shape: (45, 37)
Version: v2, Merged Shape: (0, 41)


In [20]:
# Debug which extra columns are present in each merged dataset compared to expected columns

for version, df in final_dfs.items():
    extra_columns = set(df.columns) - set(expected_columns)
    print(f"Version: {version}, Extra Columns: {extra_columns}")

Version: v1, Extra Columns: {'project_token_start_weight', 'primary_token_pair', 'project_token_address', 'peak_swaps_in_bucket', 'end_time', 'peak_activity_time_normalized', 'project_token_delta_weight', 'payment_token_symbol', 'project_token_weight_slope_per_hour', 'project_token_end_weight', 'initial_price_time', 'start_weights_symbols', 'participants', 'amount_raised_usd', 'payment_token_address', 'end_weights_symbols', 'duration_days', 'initial_price_usd', 'end_weights', 'start_time', 'project_token_weight_slope_pct_per_hour', 'lbp_crp_address', 'txns', 'duration_seconds', 'volume_usd', 'duration_hours', 'start_weights', 'project_token_symbol'}
Version: v2, Extra Columns: {'weight_slope_token0_per_hour', 'token0', 'primary_token_pair', 'peak_swaps_in_bucket', 'token1_symbol', 'schedule_end_time', 'blockchain', 'weight_slope_token1_per_hour', 'end_time', 'peak_activity_time_normalized', 'token0_symbol', 'end_weight_token0', 'net_usd_amount', 'net_token_amount', 'start_weight_token1